In [ ]:
import pandas as pd
import numpy as np
import time
import seaborn as sns
import sqlite3
# from pandas_summary import dataframesummary
from sklearn.ensemble import  RandomForestRegressor ,RandomForestClassifier
from sklearn import metrics
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error,r2_score,root_mean_squared_error
from sklearn.tree import plot_tree
from sklearn.tree import DecisionTreeRegressor,DecisionTreeClassifier 
# =============split training and validation================
def split_vals(a,n):return a[:n].copy(),a[n:].copy()
#================missing_handling====================
def missing_handling(df):
   numeric_col=df.select_dtypes(include=[np.number]).columns
   cat_col=df.select_dtypes(include=['category']).columns
   for col in numeric_col:
       df[col]=df[col].fillna(df[col].mean())
   for col in cat_col:
       if not df[col].mode().empty:
        df[col]=df[col].fillna(df[col].mode()[0])
# =============categorize it================
from pandas.api.types import is_string_dtype
def train_cats(df):
    for n,c in df.items():
        if is_string_dtype(c) or df[n].dtype=='object':
                df[n]=c.astype('category').cat.as_ordered()
# =============codec================
def numericalize(df,col,name,max_n_cat):
    if not is_numeric_dtype(col) and (max_n_cat is None or col.unique()>max_n_cat):
        df[name]=col.cat.codes+1
# =============get_samples================
def get_samples(df,size):
    return df.sample(n=size).reset_index(drop=True)
# =============codec and split it to x,y================
from pandas.api.types import is_numeric_dtype,is_string_dtype
def proc_df(df,y_fld,skip_flds=None,do_scale=False,preproc_fn=None,max_n_cat=None
            ,subset=None):
    if not skip_flds:skip_flds=[]
    if subset :df=get_samples(df,subset)
    df=df.copy()
    if preproc_fn :preproc_fn(df)
    y=df[y_fld].values
    df.drop(skip_flds+[y_fld],axis=1,inplace=True)
    # for n,c in df.items():
    #     fix_missing(df,c,n)
    if do_scale:mapper=scale_vars(df)
    for n,c in df.items():
        print(f'proccessing column:{n}')
        numericalize(df,c,n,max_n_cat)
        res=[pd.get_dummies(df,dummy_na=True),y]
    if not do_scale :return res 
  # =============summary of mse,rmse,r2 for training data================      
def print_score(m):
    res=['mse',mean_squared_error(y,m.predict(trn)),
         'mse val',mean_squared_error(y_val,m.predict(val)),
         'rmse',root_mean_squared_error(y,m.predict(trn)),
         'rmse val',root_mean_squared_error(y_val,m.predict(val)),
         'r2',m.score(trn,y),
        'r2 val data',m.score(val,y_val)]
    if hasattr(m,'oob_score_'):res.append(m.oob_score_)  
    return res

In [ ]:
import os
print(os.listdir('.'))

In [ ]:
import sys
try:
    import cgi
except ImportError:
    import legacy_cgi
    sys.modules['cgi'] = legacy_cgi

import opendatasets as od
dataset_url = "https://www.kaggle.com/datasets/kanchana1990/new-hampshire-real-estate-data-2026?select=new_hampshire_real_estate_2026.csv"
od.download(dataset_url)
file_name = 'new-hampshire-real-estate-data-2026?select=new_hampshire_real_estate_2026.csv'

if os.path.exists(file_name):
    df = pd.read_csv(file_name)
    print(df.head())
else:
    print(f"file'{file_name}' does not exist. Please check the file path and try again.")
    print("https://www.kaggle.com/datasets/aravindpcoder/instagram-usage-lifestyle-dataset")

In [ ]:
df.describe(include='all')
df.dtypes
# df

In [ ]:
len(df)

In [ ]:
df=df.drop('text',axis=1)
train_cats(df)
nval=1000 #0.15*len(df)
ntrn=len(df)-nval
df['listPrice']=np.log(df['listPrice'])
# trnall,valall=split_vals(df,ntrn)
# missing_handling(trnall) 
# missing_handling(valall)
# trn,y=proc_df(trnall.drop(['baths_full','baths_full_calc'],axis=1),'listPrice')
# val,y_val=proc_df(valall.drop(['baths_full','baths_full_calc'],axis=1),'listPrice')

missing_handling(df)
trn,y=proc_df(df.drop(['baths_full','baths_full_calc'],axis=1),'listPrice')

In [ ]:
# %prun np.array(trn)

In [ ]:
sns.histplot(y,kde=True)
plt.show()

In [ ]:
%time m=RandomForestRegressor(n_jobs=-1,oob_score=True)
m.fit(trn,y)
print(print_score(m))

In [ ]:
fi=pd.DataFrame({'cols':trn.columns,'importanc':m.feature_importances_}).sort_values('importanc',ascending=False)
# print(fi)
fi.plot('cols','importanc','barh')

<h3>estimate confidence of the model in that row (because there is row that is rare that model didn't train it or close it while training

In [ ]:
%time pred=np.stack([e.predict(val) for e in m.estimators_])
np.std(pred[:,0]),np.mean(pred[:,0]),y_val[0]

In [ ]:
def get_pred(t):
    return t.predict(val)
%time pred=np.stack(parallel_trees(m,get_pred))
np.std(pred[:,0]),np.mean(pred[:,0]),y_val[0]

from concurrent.futures import ProcessPoolExecutor
def parallel_trees(m,fn,n_workers=8):
    #m: random forest model fn:function i want to apply it on each tree n_workers:#cores 
    with processPoolExecutor(max_workers=n_workers)as ex:
        return list(ex.map(fn,m.estimators_))

In [ ]:
pred.shape

In [ ]:
trnall.type.value_counts().plot.barh()

In [ ]:
x=valall.copy()
x['pred_std']=np.std(pred,axis=0)
x['pred']=np.mean(pred,axis=0)
# v=pd.DataFrame( x.value_counts().array[-(x.shape[0]):])
# # x
x.type.value_counts().plot.barh()
print(enc)
flds=['type','listPrice','pred','pred_std']
enc=x[flds].groupby('type',as_index=False).mean()
# print(enc.sort_values('listPrice',ascending=False))
prec=(enc.pred_std/enc.pred)#.sort_values(ascending=False)
prec
# np.subtract(enc['listPrice'],enc['pred']

In [ ]:
enc.plot('type','listPrice','barh')

In [ ]:
enc.plot('type','pred','barh',xerr='pred_std')

In [ ]:
fi